# Lab — Infrastructure as Code with Bicep

**Birlasoft FORGE FDE Academy · Sprint 3 · 30 September 2026**

---

### The good news about today's lab

Almost all of it runs **with no Azure subscription and no cost at all**. The Bicep compiler, the linter, the unit tests, the what-if filter and the cost model are all just programs running here in Colab. You only need Azure for one optional read-only check near the end.

| Part | What you do | Needs Azure? | Cost |
|---|---|---|---|
| 0 | Install the Bicep compiler | No | ₹0 |
| 1 | Write three modules and compile them | No | ₹0 |
| 2 | Catch mistakes with the linter | No | ₹0 |
| 3 | Unit test your infrastructure (and find a real bug) | No | ₹0 |
| 4 | Build the what-if noise filter | No | ₹0 |
| 5 | Work out the deployment order | No | ₹0 |
| 6 | Drift: prove your template is not the truth | Optional, read-only | ₹0 |
| 7 | Build the three-figure cost model | No | ₹0 |
| 8 | Export your ADR and cost sheet | No | ₹0 |

### Three rules, same as always

1. **Use the resource group you were already given.** Part 6 only *reads* from it. We create nothing.
2. **Never type a key or password into a cell.** Nothing here asks for one — and that is itself part of the lesson.
3. **Read the output, not just the green tick.** Two cells in this notebook are *designed* to fail. Finding out why is the exercise.

## Part 0 · Setup

### Cell 1 — Install the Bicep compiler

Bicep is a single downloadable program. It does not need Azure, it does not need a login, and it does not talk to the internet once it is here. That surprises people — they assume infrastructure tooling must be connected to a cloud.

**The lesson in this cell:** Bicep is a *compiler*. It turns one text file into another text file. Everything Azure-related happens later, when somebody sends that output to Azure. That separation is what lets you test infrastructure cheaply — you can check your work hundreds of times without touching a subscription or spending a rupee.

In [1]:
import subprocess, os, json, textwrap, sys

os.makedirs("/content/iac/modules", exist_ok=True) if os.path.isdir("/content") else os.makedirs("iac/modules", exist_ok=True)
ROOT = "/content/iac" if os.path.isdir("/content") else "iac"
os.chdir(ROOT)

if not os.path.exists("bicep"):
    subprocess.run("curl -sL -o bicep https://github.com/Azure/bicep/releases/latest/download/bicep-linux-x64 && chmod +x bicep",
                   shell=True, check=True)
BICEP = os.path.abspath("bicep")

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.returncode, (r.stdout or "") + (r.stderr or "")

print(sh(BICEP + " --version")[1].strip())
print("Working directory:", os.getcwd())

Bicep CLI version 0.47.16 (3f73e1a234)
Working directory: /content/iac


## Part 1 · Write it, then compile it

### Cell 2 — Three modules, and a main file that calls them

We are building the DecisionStream infrastructure as three modules. The interesting decision is not how to write a module — it is **where to draw the line between them**.

The rule: **group by lifecycle, not by technology.** Things that change together belong together.

- **observability** — Log Analytics and Application Insights. Deployed first, removed last, because everything else logs into it.
- **data** — Storage and Cosmos DB. Changes rarely. Holds the claims. Must not be casually redeployed.
- **compute** — App Service plan and the Function app. Changes weekly.

If all of this were one file, every small change to a Function would redeploy the claims database. Eventually somebody stops deploying, and that is how IaC dies in a real project.

> Read the Bicep below before you run it. You will notice `data.bicep` has an output called `storageKey`. Leave it there. It is deliberate and we will come back to it in Part 3.

In [2]:
observability = """
param env string
param location string

resource law 'Microsoft.OperationalInsights/workspaces@2023-09-01' = {
  name: 'log-ds-${env}'
  location: location
  properties: {
    sku: { name: 'PerGB2018' }
    retentionInDays: 30
  }
}

resource appi 'Microsoft.Insights/components@2020-02-02' = {
  name: 'appi-ds-${env}'
  location: location
  kind: 'web'
  properties: {
    Application_Type: 'web'
    WorkspaceResourceId: law.id
  }
}

output workspaceId string = law.id
output appInsightsConnString string = appi.properties.ConnectionString
"""

data = """
param env string
param location string

resource sa 'Microsoft.Storage/storageAccounts@2023-01-01' = {
  name: 'stds${env}${uniqueString(resourceGroup().id)}'
  location: location
  sku: { name: 'Standard_LRS' }
  kind: 'StorageV2'
  properties: {
    allowSharedKeyAccess: false
    supportsHttpsTrafficOnly: true
    minimumTlsVersion: 'TLS1_2'
  }
}

resource cosmos 'Microsoft.DocumentDB/databaseAccounts@2024-05-15' = {
  name: 'cosmos-ds-${env}-${uniqueString(resourceGroup().id)}'
  location: location
  kind: 'GlobalDocumentDB'
  properties: {
    databaseAccountOfferType: 'Standard'
    disableLocalAuth: true
    minimalTlsVersion: 'Tls12'
    locations: [ { locationName: location, failoverPriority: 0 } ]
  }
}

output storageId string = sa.id
output cosmosId string = cosmos.id
output storageKey string = sa.listKeys().keys[0].value
"""

compute = """
param env string
param location string
param planSku string
param appInsightsConnString string
param storageName string

resource plan 'Microsoft.Web/serverfarms@2023-12-01' = {
  name: 'plan-ds-${env}'
  location: location
  sku: { name: planSku }
  kind: 'linux'
  properties: { reserved: true }
}

resource fn 'Microsoft.Web/sites@2023-12-01' = {
  name: 'func-ds-${env}-${uniqueString(resourceGroup().id)}'
  location: location
  kind: 'functionapp,linux'
  identity: { type: 'SystemAssigned' }
  properties: {
    serverFarmId: plan.id
    httpsOnly: true
    siteConfig: {
      linuxFxVersion: 'Python|3.11'
      minTlsVersion: '1.2'
      ftpsState: 'Disabled'
      appSettings: [
        { name: 'APPLICATIONINSIGHTS_CONNECTION_STRING', value: appInsightsConnString }
        { name: 'STORAGE_ACCOUNT_NAME', value: storageName }
      ]
    }
  }
}

output functionPrincipalId string = fn.identity.principalId
output functionHostName string = fn.properties.defaultHostName
"""

main = """
targetScope = 'resourceGroup'

@description('Short environment name')
@allowed([ 'dev', 'stg', 'prod' ])
param env string

@description('Azure region for every resource')
param location string = resourceGroup().location

@description('App Service plan SKU. Y1 is Consumption.')
param planSku string = 'Y1'

module obs './modules/observability.bicep' = {
  name: 'observability'
  params: { env: env, location: location }
}

module dat './modules/data.bicep' = {
  name: 'data'
  params: { env: env, location: location }
}

module cmp './modules/compute.bicep' = {
  name: 'compute'
  params: {
    env: env
    location: location
    planSku: planSku
    appInsightsConnString: obs.outputs.appInsightsConnString
    storageName: 'stds${env}${uniqueString(resourceGroup().id)}'
  }
}

output apiHostName string = cmp.outputs.functionHostName
output cosmosResourceId string = dat.outputs.cosmosId
output functionPrincipalId string = cmp.outputs.functionPrincipalId
"""

for name, body in [("modules/observability.bicep", observability),
                   ("modules/data.bicep", data),
                   ("modules/compute.bicep", compute),
                   ("main.bicep", main)]:
    open(name, "w").write(body.strip() + "\n")
    print("wrote", name, len(body.strip().split(chr(10))), "lines")

wrote modules/observability.bicep 24 lines
wrote modules/data.bicep 30 lines
wrote modules/compute.bicep 36 lines
wrote main.bicep 36 lines


### Cell 3 — Compile it, and look at what came out

Azure does not understand Bicep. It only ever understands ARM JSON. Bicep is a friendlier language that **compiles down** to that JSON.

This matters practically: whenever Bicep behaves strangely, compiling it and reading the JSON is usually how you find out why.

**The lesson in this cell:** watch the size ratio. Your readable Bicep becomes a much larger JSON document. That ratio is the entire reason Bicep exists — nobody can review the JSON version in a pull request, and a thing nobody reviews is a thing nobody catches mistakes in.

In [3]:
rc, out = sh(BICEP + " build main.bicep --outfile main.json")
print("exit code:", rc)
print(out if out.strip() else "(compiled with no messages)")

if rc == 0:
    bicep_lines = sum(len(open(f).read().split(chr(10)))
                      for f in ["main.bicep", "modules/observability.bicep",
                                "modules/data.bicep", "modules/compute.bicep"])
    arm = open("main.json").read()
    print()
    print("Your Bicep :", bicep_lines, "lines across 4 files")
    print("The ARM    :", len(arm.split(chr(10))), "lines in 1 file")
    print("Ratio      :", round(len(arm.split(chr(10))) / bicep_lines, 1), "x bigger")
    print()
    print("That ratio is why Bicep exists. Nobody reviews the second one.")

exit code: 0
/content/iac/modules/data.bicep(30,28) : Warning outputs-should-not-contain-secrets: Outputs should not contain secrets. Found possible secret: function 'listKeys' [https://aka.ms/bicep/linter-diagnostics#outputs-should-not-contain-secrets]


Your Bicep : 130 lines across 4 files
The ARM    : 333 lines in 1 file
Ratio      : 2.6 x bigger

That ratio is why Bicep exists. Nobody reviews the second one.


### Cell 4 — What happened to your modules?

Open the compiled JSON and you will find something worth understanding: **each module became a nested deployment** of type `Microsoft.Resources/deployments`.

That single fact explains several things you will meet later:

- Modules show up as separate deployments in the portal's deployment history.
- The `what-if` preview has limits counted in *nested templates*, which are exactly these.
- Module outputs are passed between deployments, which is why a secret in an output is stored in deployment history — the history of a real deployment, not some abstract log.

In [4]:
doc = json.loads(open("main.json").read())

print("Top-level resources in the compiled ARM:")
for r in doc["resources"]:
    print("  " + r["type"].ljust(38) + r.get("name", "?"))

print()
print("Parameters exposed:", list(doc.get("parameters", {}).keys()))
print("Outputs exposed   :", list(doc.get("outputs", {}).keys()))
print()
print("Notice: the storage account and Cosmos DB are NOT in this list.")
print("They are inside the nested 'data' deployment. Azure will unpack it at deploy time.")

Top-level resources in the compiled ARM:
  Microsoft.Resources/deployments       observability
  Microsoft.Resources/deployments       data
  Microsoft.Resources/deployments       compute

Parameters exposed: ['env', 'location', 'planSku']
Outputs exposed   : ['apiHostName', 'cosmosResourceId', 'functionPrincipalId']

Notice: the storage account and Cosmos DB are NOT in this list.
They are inside the nested 'data' deployment. Azure will unpack it at deploy time.


## Part 2 · The linter

### Cell 5 — Turn the rules up, then break something on purpose

Bicep has a linter built in. It runs as you type in VS Code and again when you compile. By default most rules are only warnings, and warnings get ignored — so the first thing to do on a real project is decide which rules should **fail the build**.

You set that in `bicepconfig.json`, which lives next to your Bicep files.

**The lesson in this cell:** a warning nobody has to fix is a warning nobody fixes. Choosing your blocking rule set is a team decision, and making it early is much easier than making it after three months of accumulated warnings.

> One rule below is deliberately left as a **warning** rather than an error: `outputs-should-not-contain-secrets`. That is so you can watch your own unit test catch the same problem in Part 3, at a strength the linter did not have. On a real project you turn it up to error — and we do, at the end of Part 3.

In [5]:
bicepconfig = {
  "analyzers": {
    "core": {
      "enabled": True,
      "rules": {
        "no-hardcoded-env-urls":      { "level": "error" },
        "no-unused-params":           { "level": "error" },
        "no-unused-vars":             { "level": "error" },
        "secure-parameter-default":   { "level": "error" },
        "outputs-should-not-contain-secrets": { "level": "warning" },
        "no-hardcoded-location":      { "level": "warning" },
        "prefer-interpolation":       { "level": "warning" }
      }
    }
  }
}
open("bicepconfig.json", "w").write(json.dumps(bicepconfig, indent=2))
print("bicepconfig.json written. Rule levels:")
for k, v in bicepconfig["analyzers"]["core"]["rules"].items():
    print("  " + k.ljust(40) + v["level"])
print()
print("Note one deliberate choice: outputs-should-not-contain-secrets is set to")
print("WARNING, not error. That is on purpose for this lab - in Part 3 you will")
print("watch your own test catch the same problem the linter only whispered about.")
print("We turn it up to error at the end, once the template is clean.")

bicepconfig.json written. Rule levels:
  no-hardcoded-env-urls                   error
  no-unused-params                        error
  no-unused-vars                          error
  secure-parameter-default                error
  outputs-should-not-contain-secrets      warning
  no-hardcoded-location                   warning
  prefer-interpolation                    warning

Note one deliberate choice: outputs-should-not-contain-secrets is set to
WARNING, not error. That is on purpose for this lab - in Part 3 you will
watch your own test catch the same problem the linter only whispered about.
We turn it up to error at the end, once the template is clean.


### Cell 6 — Watch it catch four classic mistakes

The file below contains four things that look completely reasonable and are all wrong:

1. A **default value on a secure parameter** — the whole point of `@secure()` is that the value comes from outside. A default is a hardcoded credential with extra steps.
2. A **hardcoded management URL** — this breaks the moment the client deploys to a sovereign or government cloud.
3. An **unused parameter** — harmless-looking, but it usually means somebody removed a resource and forgot the wiring, so it is a signal that something else is missing too.
4. **String concatenation** instead of interpolation — only a style issue, which is why it is a warning and not an error.

In [6]:
bad = """
@secure()
param adminPassword string = 'P@ssw0rd123!'

param unusedThing string

param storageName string

var portalUrl = 'https://management.azure.com/subscriptions'

resource sa 'Microsoft.Storage/storageAccounts@2023-01-01' = {
  name: concat('st', storageName)
  location: 'centralindia'
  sku: { name: 'Standard_LRS' }
  kind: 'StorageV2'
  properties: {
    supportsHttpsTrafficOnly: true
    minimumTlsVersion: 'TLS1_2'
  }
}

output where string = '${portalUrl}/${adminPassword}'
"""
open("bad.bicep", "w").write(bad.strip() + "\n")

rc, out = sh(BICEP + " build bad.bicep --outfile /dev/null")
print("exit code:", rc, "  (non-zero means the build FAILED, which is what we want)\n")
for line in out.split(chr(10)):
    if "Error" in line or "Warning" in line:
        print(" ", line.strip()[:160])

print()
print("Every one of those would have shipped without the config file from Cell 5.")
print("The linter is free. The only decision you have to make is which rules block.")

exit code: 1   (non-zero means the build FAILED, which is what we want)

  /content/iac/bad.bicep(2,28) : Error secure-parameter-default: Secure parameters should not have hardcoded defaults (except for empty or newGuid()). [https://ak
  /content/iac/bad.bicep(4,7) : Error no-unused-params: Parameter "unusedThing" is declared but never used. [https://aka.ms/bicep/linter-diagnostics#no-unused-par
  /content/iac/bad.bicep(8,26) : Error no-hardcoded-env-urls: Environment URLs should not be hardcoded. Use the environment() function to ensure compatibility acr
  /content/iac/bad.bicep(11,9) : Warning prefer-interpolation: Use string interpolation instead of the concat function. [https://aka.ms/bicep/linter-diagnostics#p
  /content/iac/bad.bicep(12,13) : Warning no-hardcoded-location: A resource location should not use a hard-coded string or variable value. Please use a parameter 
  /content/iac/bad.bicep(21,39) : Warning outputs-should-not-contain-secrets: Outputs should not contain secrets

## Part 3 · Unit tests — and a real bug in your own template

### Cell 7 — Be precise about what a Bicep test proves

The brief asks for Bicep unit tests in pytest. That is a real and useful practice, but be exact about the claim it supports.

A Bicep unit test **compiles your Bicep to ARM and asserts on the JSON you produced.** It tells you *the template you generated is the template you meant*. It says nothing at all about whether Azure will accept it, or what you will actually get.

That is a genuinely different claim from an application unit test, and the difference matters:

| Level | What it tests | Catches | Cost |
|---|---|---|---|
| Lint | Your Bicep | Style, unused params, bad defaults | Instant |
| **Compile + assert** | **Your template** | **Wrong SKU, missing property, bad wiring** | **Seconds** |
| What-if | Azure's prediction | Unexpected deletes, replacements | ~1 min |
| Ephemeral deploy | The actual cloud | Everything the others miss | Minutes, and money |
| Smoke test | The running system | That it actually works | Minutes |

Only the bottom two rows test the cloud. Both halves are worth doing — but only one of them answers "will this work?".

In [7]:
tests = '''
import json, subprocess, os, pytest

HERE = os.path.dirname(os.path.abspath(__file__))

@pytest.fixture(scope="session")
def arm():
    subprocess.run([os.path.join(HERE, "bicep"), "build",
                    os.path.join(HERE, "main.bicep"),
                    "--outfile", os.path.join(HERE, "test_main.json")], check=True)
    return json.load(open(os.path.join(HERE, "test_main.json")))

def nested(arm, module_name):
    """Pull the inner template out of one nested module deployment."""
    for r in arm["resources"]:
        if r["type"] == "Microsoft.Resources/deployments" and r["name"] == module_name:
            return r["properties"]["template"]
    raise AssertionError("module not found: " + module_name)

def resources_of(tpl, rtype):
    return [r for r in tpl["resources"] if r["type"].lower() == rtype.lower()]


def test_env_parameter_is_restricted(arm):
    """A free-text environment name lets somebody deploy to 'prdo'."""
    assert arm["parameters"]["env"]["allowedValues"] == ["dev", "stg", "prod"]

def test_storage_refuses_shared_keys(arm):
    """Your Day 42 decision, now enforced by a test that runs on every commit."""
    sa = resources_of(nested(arm, "data"), "Microsoft.Storage/storageAccounts")[0]
    assert sa["properties"]["allowSharedKeyAccess"] is False

def test_cosmos_refuses_local_auth(arm):
    cos = resources_of(nested(arm, "data"), "Microsoft.DocumentDB/databaseAccounts")[0]
    assert cos["properties"]["disableLocalAuth"] is True

def test_function_has_managed_identity(arm):
    """No identity means the app will end up using a key instead."""
    fn = resources_of(nested(arm, "compute"), "Microsoft.Web/sites")[0]
    assert fn["identity"]["type"] == "SystemAssigned"

def test_function_is_https_only(arm):
    fn = resources_of(nested(arm, "compute"), "Microsoft.Web/sites")[0]
    assert fn["properties"]["httpsOnly"] is True

def test_no_output_anywhere_contains_a_secret(arm):
    """Deployment outputs are stored in the resource group history, in plain text,
    readable by anyone with Reader, for months. Rotating the key does NOT
    remove the old value from that history."""
    danger = ("listkeys", "listsecrets", "listconnectionstrings",
              "password", "secretvalue", "accountkey")
    offenders = []

    def scan(where, outputs):
        for name, spec in (outputs or {}).items():
            blob = json.dumps(spec).lower()
            if any(d in blob for d in danger) or "key" == name.lower()[-3:]:
                offenders.append(where + " -> " + name)

    scan("main", arm.get("outputs"))
    for r in arm["resources"]:
        if r["type"] == "Microsoft.Resources/deployments":
            scan(r["name"], r["properties"]["template"].get("outputs"))

    assert not offenders, "Secret-bearing outputs found: " + ", ".join(offenders)
'''
open("test_infra.py", "w").write(tests.strip() + "\n")
print("test_infra.py written -", tests.count("def test_"), "tests")

test_infra.py written - 6 tests


### Cell 8 — Run the tests. One of them is going to fail.

Run this and read the failure carefully. It is not a mistake in the notebook — it is a real problem in the template you wrote in Cell 2, and it is the single most common way a secret leaks out of an Azure deployment.

Think about what the fix should be before you look at the next cell.

In [8]:
!pip -q install pytest 2>/dev/null
rc, out = sh("python -m pytest test_infra.py -q --no-header --tb=line 2>&1 | tail -14")
print(out)
print("=" * 70)
print("Five passed. One FAILED. Read the failure line above before continuing -")
print("it names the exact output that is leaking, and which module it is in.")

.....F                                                                   [100%]
=================================== FAILURES ===================================
/content/iac/test_infra.py:64: AssertionError: Secret-bearing outputs found: data -> storageKey
=========================== short test summary info ============================
FAILED test_infra.py::test_no_output_anywhere_contains_a_secret - AssertionEr...
1 failed, 5 passed in 3.04s

Five passed. One FAILED. Read the failure line above before continuing -
it names the exact output that is leaking, and which module it is in.


### Cell 9 — Why that failure matters more than it looks

The offending line is in `data.bicep`:

```bicep
output storageKey string = sa.listKeys().keys[0].value
```

It looks helpful. The pipeline needs the key, so you output the key. Here is what actually happens:

- Deployment outputs are saved in the **resource group's deployment history**.
- They are stored **in plain text**.
- **Anyone with Reader** on the resource group can list deployments and read them.
- They stay there for **months**.
- And **rotating the key does not clean the history** — the old value stays, next to a record of exactly which storage account it belonged to.

So this one line published a live credential to every person who can see the subscription, and left a permanent copy that survives the fix.

**The distinction to hold on to:** a Key Vault *secret URI* is safe to output — it is an address, and reaching it still needs a role, which is exactly your Day 42 work. The *secret value* is never safe. Same word, completely different object.

In [9]:
src = open("modules/data.bicep").read()
fixed = src.replace("output storageKey string = sa.listKeys().keys[0].value\n", "")
fixed = fixed.rstrip() + "\noutput storageName string = sa.name\n"
open("modules/data.bicep", "w").write(fixed)

print("Removed the storageKey output. Replaced it with the account NAME, which is not a secret.")
print("Anything that needs the key should use managed identity instead - see Day 42.\n")

rc, out = sh("python -m pytest test_infra.py -q --no-header 2>&1 | tail -4")
print(out)

# Now that the template is clean, raise the linter rule to error so it can never come back.
bicepconfig["analyzers"]["core"]["rules"]["outputs-should-not-contain-secrets"]["level"] = "error"
open("bicepconfig.json", "w").write(json.dumps(bicepconfig, indent=2))
rc2, out2 = sh(BICEP + " build main.bicep --outfile main.json")
print("Linter rule raised to ERROR. Rebuild exit code:", rc2,
      "(0 means the template is clean at the stricter level)")
print()
print("That is the order to do this in on a real project: fix the template first,")
print("THEN raise the rule. Raising it first just gives the team a red build they")
print("did not ask for, and the usual response is to turn the rule back off.")

Removed the storageKey output. Replaced it with the account NAME, which is not a secret.
Anything that needs the key should use managed identity instead - see Day 42.

......                                                                   [100%]
6 passed in 2.48s

Linter rule raised to ERROR. Rebuild exit code: 0 (0 means the template is clean at the stricter level)

That is the order to do this in on a real project: fix the template first,
THEN raise the rule. Raising it first just gives the team a red build they
did not ask for, and the usual response is to turn the rule back off.


## Part 4 · The what-if noise filter

### Cell 10 — Why what-if needs a filter and not a reader
### The Azure Resource Manager (ARM) What-If operation lets you preview changes before you deploy an ARM or Bicep template
`what-if` asks Azure to predict what a deployment would change. It is the best preview available and you should always run it. It is also wrong in specific, **documented** ways. Microsoft's own documentation says so.

What-if cannot evaluate:

| It cannot see | So you get | Why it matters for you |
|---|---|---|
| `listKeys()` and similar | A change, every single time | Every key or connection-string line |
| `reference()` to other resources | A change, every single time | Cross-module wiring always looks dirty |
| Secure parameters | Never shown | Your secrets path is invisible to review |
| `utcNow()`, `newGuid()` | The literal function text | Anything tagged with a deploy time |
| Past 500 nested templates / 5 min | Change type becomes `Ignore` | Large estates quietly stop being checked |

Microsoft's own word for the result is **"noise"**. Properties get reported as deleted when they will not change, because they are absent from your Bicep but set to a default during deployment.

**So a what-if output full of alarming red is the normal, healthy case.** And that is exactly what makes it dangerous — which is the next cell.

In [10]:
sample_whatif = {
 "status": "Succeeded",
 "properties": {"changes": [
  {"changeType": "Modify", "resourceId": ".../sites/func-ds-prod",
   "delta": [{"path": "properties.siteConfig.appSettings[0].value",
              "propertyChangeType": "Modify",
              "before": "InstrumentationKey=abc...",
              "after": "[reference(resourceId('Microsoft.Insights/components','appi-ds-prod')).ConnectionString]"}]},
  {"changeType": "Modify", "resourceId": ".../storageAccounts/stdsprod",
   "delta": [{"path": "properties.encryption.keySource",
              "propertyChangeType": "Delete",
              "before": "Microsoft.Storage", "after": None}]},
  {"changeType": "Modify", "resourceId": ".../storageAccounts/stdsprod",
   "delta": [{"path": "tags.lastDeployed", "propertyChangeType": "Modify",
              "before": "20260901T101500Z", "after": "[utcNow()]"}]},
  {"changeType": "NoChange", "resourceId": ".../workspaces/log-ds-prod", "delta": []},
  {"changeType": "Modify", "resourceId": ".../databaseAccounts/cosmos-ds-prod",
   "delta": [{"path": "properties.disableLocalAuth",
              "propertyChangeType": "Modify", "before": True, "after": False}]},
  {"changeType": "Delete", "resourceId": ".../databaseAccounts/cosmos-ds-prod/sqlDatabases/claims",
   "delta": []},
  {"changeType": "Deploy", "resourceId": ".../serverfarms/plan-ds-prod", "delta": []},
  {"changeType": "Ignore", "resourceId": ".../service/apim-ds-prod", "delta": []}
 ]}
}
json.dump(sample_whatif, open("whatif.json", "w"), indent=1)

print("A realistic what-if result:", len(sample_whatif["properties"]["changes"]), "changes.")
print()
for c in sample_whatif["properties"]["changes"]:
    print("  " + c["changeType"].ljust(10) + c["resourceId"].split("/")[-1])
print()
print("Six of these are noise or nothing. Two would take production down.")
print("Reading this as prose, at 11pm, you would miss them. So do not read it as prose.")

A realistic what-if result: 8 changes.

  Modify    func-ds-prod
  Modify    stdsprod
  Modify    stdsprod
  NoChange  log-ds-prod
  Modify    cosmos-ds-prod
  Delete    claims
  Deploy    plan-ds-prod
  Ignore    apim-ds-prod

Six of these are noise or nothing. Two would take production down.
Reading this as prose, at 11pm, you would miss them. So do not read it as prose.


### Cell 11 — Stop asking humans to filter noise

Here is how the accident happens, and everyone in the room has lived through some version of it:

- **Week 1** — the team reads every line of the what-if output.
- **Week 3** — they have learned that most of the red lines are noise.
- **Week 6** — somebody skims, sees the usual red, and approves.
- **Week 6** — that one was a real delete of a Cosmos container.

A tool that cries wolf forty times teaches you not to look on the forty-first. The fix is not to ask people to concentrate harder. **The fix is to narrow the check** so that a human only looks when something is actually worth looking at.

A machine never gets bored of noise. Give it the boring job.

In [11]:
NOISE_MARKERS = ("[reference(", "[listkeys(", "[utcnow(", "[newguid(", "[list")
DEFAULTY = {"microsoft.storage", "standard", "enabled", "default"}
IMMUTABLE = ("partitionkey", "keyvaultkeyuri", "location", "kind")

def is_noise(d):
    after = str(d.get("after") or "").lower()
    before = str(d.get("before") or "").lower()
    if any(m in after for m in NOISE_MARKERS):
        return True, "unresolved ARM expression - what-if cannot evaluate it"
    if d.get("propertyChangeType") == "Delete" and before in DEFAULTY:
        return True, "a service default absent from your Bicep - documented noise"
    return False, ""

def classify(change):
    ct = change["changeType"]
    if ct == "Delete":
        return "FATAL", "resource would be DELETED"
    if ct == "Deploy":
        return "REVIEW", "what-if could not determine the change"
    if ct in ("NoChange", "Ignore"):
        return "quiet", ct
    real = []
    for d in change.get("delta", []):
        noise, _ = is_noise(d)
        if not noise:
            if any(k in d["path"].lower() for k in IMMUTABLE):
                return "FATAL", "immutable property change -> replacement: " + d["path"]
            real.append(d["path"])
    return ("REAL", ", ".join(real)) if real else ("noise", "all property changes are known noise")

verdicts = {}
print("VERDICT".ljust(9) + "RESOURCE".ljust(34) + "WHY")
print("-" * 96)
for c in sample_whatif["properties"]["changes"]:
    v, why = classify(c)
    verdicts[v] = verdicts.get(v, 0) + 1
    print(v.ljust(9) + c["resourceId"].split("/")[-1][:32].ljust(34) + why[:50])

fatal = verdicts.get("FATAL", 0)
print()
print(verdicts)
print()
print("PIPELINE RESULT:", "FAIL - a human must look" if fatal else "pass")
print()
print("Two lines needed a human. Six did not. That is the whole idea:")
print("the human now reads a two-line report instead of skimming a forty-line one.")

VERDICT  RESOURCE                          WHY
------------------------------------------------------------------------------------------------
noise    func-ds-prod                      all property changes are known noise
noise    stdsprod                          all property changes are known noise
noise    stdsprod                          all property changes are known noise
quiet    log-ds-prod                       NoChange
REAL     cosmos-ds-prod                    properties.disableLocalAuth
FATAL    claims                            resource would be DELETED
REVIEW   plan-ds-prod                      what-if could not determine the change
quiet    apim-ds-prod                      Ignore

{'noise': 3, 'quiet': 2, 'REAL': 1, 'FATAL': 1, 'REVIEW': 1}

PIPELINE RESULT: FAIL - a human must look

Two lines needed a human. Six did not. That is the whole idea:
the human now reads a two-line report instead of skimming a forty-line one.


### Cell 12 — And notice which one it caught

Look at the Cosmos DB line in the output above. It was classified **REAL**, not noise:

```
properties.disableLocalAuth:  true  ->  false
```

That is somebody's deployment quietly switching key-based authentication **back on** for the claims database — undoing the Day 42 work, on a Tuesday, in a pull request that looked like a small change.

It would be one red line among forty in an unfiltered what-if. Nobody would see it.

This is the argument for the filter in one example: it is not about saving time. It is about the difference between a check that happens and a check that only appears to happen.

In [12]:
print("How you would wire this into a pipeline:\n")
print("  az deployment group what-if -g $RG -f main.bicep -p prod.bicepparam \\")
print("      --no-pretty-print > whatif.json")
print("  python check_whatif.py whatif.json      # exit 1 on any FATAL")
print()
print("And what the pipeline does with the result:\n")
print("  FATAL  -> stop, require a named human approval with a written reason")
print("  REVIEW -> stop on production, warn on dev")
print("  REAL   -> show in the PR comment, do not block")
print("  noise  -> never shown to anybody, ever")
print()
print("Write the exit code yourself:")
print("  sys.exit(1 if fatal else 0)")

How you would wire this into a pipeline:

  az deployment group what-if -g $RG -f main.bicep -p prod.bicepparam \
      --no-pretty-print > whatif.json
  python check_whatif.py whatif.json      # exit 1 on any FATAL

And what the pipeline does with the result:

  FATAL  -> stop, require a named human approval with a written reason
  REVIEW -> stop on production, warn on dev
  REAL   -> show in the PR comment, do not block
  noise  -> never shown to anybody, ever

Write the exit code yourself:
  sys.exit(1 if fatal else 0)


## Part 5 · Deployment order

### Cell 13 — Bicep works out the order by itself. Mostly.

You never write deployment order in Bicep. Azure builds a dependency graph from the *references between resources* — if module B uses `A.outputs.something`, then A must go first, and Bicep records that automatically.

This cell reads the compiled ARM and reconstructs that graph, so you can see what Azure is going to do.

**The lesson:** the dependencies Bicep works out for itself are never the problem. The problems are the ones it *cannot* see.

> **Before you run it, predict the answer.** Three modules: observability, data, compute. Which waits for which? Write it down, then run the cell and compare. If your prediction and the output disagree, the output is right and there is something to learn — which is exactly what happened when this lab was being built.

In [13]:
sh(BICEP + " build main.bicep --outfile main.json")
doc = json.loads(open("main.json").read())

deps = {}
for r in doc["resources"]:
    nm = r.get("name", "?")
    raw = r.get("dependsOn", [])
    parents = []
    for d in raw:
        for other in doc["resources"]:
            on = other.get("name", "")
            if on and on != nm and on in str(d):
                parents.append(on)
    deps[nm] = sorted(set(parents))

done, wave = set(), 1
while len(done) < len(deps):
    ready = [n for n, ps in deps.items() if n not in done and all(p in done for p in ps)]
    if not ready:
        print("CYCLE DETECTED among:", [n for n in deps if n not in done]); break
    print("Wave " + str(wave) + " (these deploy in parallel): " + ", ".join(sorted(ready)))
    for r0 in sorted(ready):
        if deps[r0]:
            print("        " + r0 + "  waits for  " + ", ".join(deps[r0]))
    done.update(ready); wave += 1

print()
print("=" * 72)
print("NOW LOOK CLOSELY AT THAT OUTPUT. Something is missing.")
print("=" * 72)
print("'compute' waits for 'observability', because it uses obs.outputs.appInsightsConnString.")
print("But 'compute' does NOT wait for 'data' - even though the Function is configured")
print("with the storage account name.")
print()
print("Why? Go back and look at main.bicep. We passed the storage name as:")
print("    storageName: 'stds${env}${uniqueString(resourceGroup().id)}'")
print("...rebuilding the same expression by hand, instead of using dat.outputs.storageName.")
print()
print("Bicep can only see a dependency when one resource REFERENCES another. Copy the")
print("expression instead of referencing the output and the edge disappears - so ARM")
print("happily deploys the Function in parallel with the storage account it points at.")
print()
print("This is a real bug in the template you wrote in Cell 2, and the dependency graph")
print("is what exposed it. FIX: pass dat.outputs.storageName, and the edge comes back.")
print()
print("Dependencies Bicep CANNOT see even when you wire it correctly:")
print("  - Role assignments are eventually consistent. A Function that starts right")
print("    after its role is granted can still get a 403. Add a retry in the smoke test.")
print("  - API Management takes 30-45 minutes. Longer than your class, and longer than")
print("    most pipeline timeouts. Use the Consumption tier, or deploy it separately.")

Wave 1 (these deploy in parallel): data, observability
Wave 2 (these deploy in parallel): compute
        compute  waits for  observability

NOW LOOK CLOSELY AT THAT OUTPUT. Something is missing.
'compute' waits for 'observability', because it uses obs.outputs.appInsightsConnString.
But 'compute' does NOT wait for 'data' - even though the Function is configured
with the storage account name.

Why? Go back and look at main.bicep. We passed the storage name as:
    storageName: 'stds${env}${uniqueString(resourceGroup().id)}'
...rebuilding the same expression by hand, instead of using dat.outputs.storageName.

Bicep can only see a dependency when one resource REFERENCES another. Copy the
expression instead of referencing the output and the edge disappears - so ARM
happily deploys the Function in parallel with the storage account it points at.

This is a real bug in the template you wrote in Cell 2, and the dependency graph
is what exposed it. FIX: pass dat.outputs.storageName, and the edg

## Part 6 · Drift — proving your template is not the truth

### Cell 14 — Incremental mode never deletes

The default deployment mode is **Incremental**, and incremental only ever adds and updates. It **never removes**.

So if you delete a resource from your Bicep file and redeploy, the resource is still there — running, costing money, and now completely undocumented, because the only record of it was the file you just deleted it from.

Your template is therefore a guarantee of a **minimum**, not a description of the whole. This cell shows it happening.

In [14]:
class ResourceGroup:
    def __init__(self): self.live = {}
    def deploy(self, template, mode="Incremental"):
        for name, props in template.items():
            self.live[name] = props
        if mode == "Complete":
            for name in list(self.live):
                if name not in template:
                    del self.live[name]
        return sorted(self.live)

v1 = {"log-ds-prod": {}, "stdsprod": {}, "cosmos-ds-prod": {}, "old-cache-redis": {"cost": 4200}}
v2 = {"log-ds-prod": {}, "stdsprod": {}, "cosmos-ds-prod": {}}   # redis removed from the Bicep

rg = ResourceGroup()
print("Deploy v1 (4 resources):      ", rg.deploy(v1))
print()
print("Somebody removes the redis cache from main.bicep. The pull request is approved.")
print("Deploy v2 (3 in the template):", rg.deploy(v2))
print()
gap = set(rg.live) - set(v2)
print("Resources running that are NOT in your template:", sorted(gap))
print("Monthly cost of that gap: Rs", sum(v.get("cost", 0) for k, v in rg.live.items() if k in gap))
print()
print("The deployment said Succeeded. Nothing warned anybody. The cache is still billing.")
print()
rg2 = ResourceGroup(); rg2.deploy(v1)
print("The same thing in COMPLETE mode:", rg2.deploy(v2, mode="Complete"))
print("Complete mode does delete it - and would also delete anything ELSE in the group")
print("that is not in your template. Point it at the wrong resource group once and")
print("the outage is total. This is why almost nobody uses it.")

Deploy v1 (4 resources):       ['cosmos-ds-prod', 'log-ds-prod', 'old-cache-redis', 'stdsprod']

Somebody removes the redis cache from main.bicep. The pull request is approved.
Deploy v2 (3 in the template): ['cosmos-ds-prod', 'log-ds-prod', 'old-cache-redis', 'stdsprod']

Resources running that are NOT in your template: ['old-cache-redis']
Monthly cost of that gap: Rs 4200

The deployment said Succeeded. Nothing warned anybody. The cache is still billing.

The same thing in COMPLETE mode: ['cosmos-ds-prod', 'log-ds-prod', 'stdsprod']
Complete mode does delete it - and would also delete anything ELSE in the group
that is not in your template. Point it at the wrong resource group once and
the outage is total. This is why almost nobody uses it.


### Cell 15 — The check that costs nothing: template versus reality

The honest answer to drift is not a tool, it is a **comparison you run on a schedule**: what is in the template, against what is actually in the resource group.

This is read-only. It creates nothing, changes nothing and costs nothing. Run it nightly on a real engagement.

**How to sign in.** We use device-code sign-in, the same as the Days 41–42 lab. It prints a short code, you open a page, you sign in normally. Notice again that there is **no key, no password and no connection string** anywhere in this notebook — your own identity is the only credential.

**To find your Subscription ID:** portal.azure.com → search **Subscriptions** → copy the long ID.
**To find your Resource Group name:** search **Resource groups** → copy the one you were given.

Leave both blank to run the same comparison against sample data.

In [15]:
SUBSCRIPTION_ID = ""    # paste your Subscription ID, or leave blank for sample mode
RESOURCE_GROUP  = ""    # paste your Resource Group name

LIVE = bool(SUBSCRIPTION_ID.strip() and RESOURCE_GROUP.strip())

def template_resource_types(armdoc):
    found = []
    def walk(tpl):
        for r in tpl.get("resources", []):
            if r["type"] == "Microsoft.Resources/deployments":
                walk(r["properties"]["template"])
            else:
                found.append(r["type"])
    walk(armdoc)
    return sorted(set(found))

in_template = template_resource_types(doc)

if LIVE:
    !pip -q install azure-identity requests
    import requests
    from azure.identity import DeviceCodeCredential
    cred = DeviceCodeCredential(client_id="04b07795-8ddb-461a-bbee-02f9e1bf7b46")
    tok = cred.get_token("https://management.azure.com/.default").token
    r = requests.get("https://management.azure.com/subscriptions/" + SUBSCRIPTION_ID +
                     "/resourceGroups/" + RESOURCE_GROUP + "/resources",
                     headers={"Authorization": "Bearer " + tok},
                     params={"api-version": "2021-04-01"}, timeout=60)
    in_cloud = sorted({x["type"] for x in r.json().get("value", [])})
else:
    in_cloud = ["Microsoft.DocumentDB/databaseAccounts", "Microsoft.Insights/components",
                "Microsoft.OperationalInsights/workspaces", "Microsoft.Storage/storageAccounts",
                "Microsoft.Web/serverfarms", "Microsoft.Web/sites",
                "Microsoft.Cache/redis", "Microsoft.Network/networkSecurityGroups"]
    print("Sample mode.\n")

only_cloud = [t for t in in_cloud if t not in in_template]
only_tpl   = [t for t in in_template if t not in in_cloud]

print("In the template AND in the cloud :", len([t for t in in_template if t in in_cloud]))
print()
print("IN THE CLOUD BUT NOT IN YOUR TEMPLATE  <-- this is the drift")
for t in only_cloud: print("   !", t)
if not only_cloud: print("   (none - but see the caution below)")
print()
print("IN YOUR TEMPLATE BUT NOT DEPLOYED HERE")
for t in only_tpl: print("   -", t)
if not only_tpl: print("   (none)")
print()
print("CAUTION: this compares resource TYPES, not individual resources or their settings.")
print("A real drift job compares resource IDs and properties too. Type-level is the")
print("cheap version that catches the biggest problem - a whole service nobody owns.")

Sample mode.

In the template AND in the cloud : 6

IN THE CLOUD BUT NOT IN YOUR TEMPLATE  <-- this is the drift
   ! Microsoft.Cache/redis
   ! Microsoft.Network/networkSecurityGroups

IN YOUR TEMPLATE BUT NOT DEPLOYED HERE
   (none)

CAUTION: this compares resource TYPES, not individual resources or their settings.
A real drift job compares resource IDs and properties too. Type-level is the
cheap version that catches the biggest problem - a whole service nobody owns.


## Part 7 · Cost

### Cell 16 — Infracost, and how to get the key

Infracost reads a template and prices it, with **no Azure credentials and nothing deployed**. It does support Bicep, with one catch: **it will not compile your Bicep for you.** Point it at a `.bicep` file and nothing useful happens. Compile first, then point it at the output folder.

**How to get an API key (free):**
1. Go to [infracost.io](https://www.infracost.io) and sign up — the free tier is fine for this.
2. Install the CLI: `curl -fsSL https://raw.githubusercontent.com/infracost/infracost/master/scripts/install.sh | sh`
3. Run `infracost auth login`, which opens a browser and stores the key locally.
4. Or set it yourself: `export INFRACOST_API_KEY=...`

The key is a secret, so it goes in an environment variable or your pipeline's secret store — never in a notebook cell and never in git.

**Documented limits worth knowing before you promise a client a number:** Bicep must be compiled first · `@metadata` decorators are lost in compilation · resources inside Template Specs are **not** expanded or priced · public-cloud pricing only.

In [ ]:
print("The exact commands, for when you have a key:\n")
for c in [
 "# 1. compile first - Infracost will NOT do this for you",
 "az bicep build --file main.bicep --outfile out/main.json",
 "",
 "# 2. price the whole thing",
 "infracost breakdown --path out/ --format table",
 "",
 "# 3. in a pull request, price only the CHANGE",
 "infracost diff --path out/ --compare-to baseline.json",
 "",
 "# a second opinion is cheap - ACE reads Bicep directly and uses the Retail Prices API",
 "azure-cost-estimator main.bicep <subscription-id> <resource-group>",
]:
    print("  " + c)
print()
print("Running the diff on every pull request is the habit worth building. A comment")
print("saying 'this change adds Rs 3,400 a month' turns cost into part of code review")
print("instead of a quarterly surprise.")

### Cell 17 — The correction that protects you in month three

The brief asks for a **monthly infrastructure cost** from Infracost. That is a real and useful number. It is also, for a system like DecisionStream, very likely **the smaller half of the bill**.

| Cost driver | In the Bicep file? | Priced by Infracost? | What actually decides it |
|---|---|---|---|
| App Service plan, APIM tier | Yes | Yes | Your SKU choice |
| Provisioned Cosmos RUs | Yes | Yes | Your throughput setting |
| Cosmos serverless / autoscale | Partly | Poorly | Real query volume |
| **Model tokens (in + out)** | **No** | **No** | **Claim volume × prompt size** |
| **Function executions** | **No** | **No** | **Traffic** |
| **Egress, storage growth, logs** | **No** | **No** | **Time and usage** |

A tool that prices the template is answering a real question — but not the one the client asked. The client asked what it will **cost to run**. The tool answered what it costs to **exist**.

So the honest deliverable is three figures and a list, not one number. The next cell builds it.

> Every price below is a **placeholder**. Put your client's real region and negotiated rates in before you show this to anybody.

In [ ]:
# ---- PLACEHOLDER PRICES. Replace with your client's region and rates. ----
FIXED = {"App Service plan (EP1)": 11500, "API Management (Developer)": 4200,
         "Cosmos DB (400 RU/s provisioned)": 1900, "Log Analytics (base)": 800,
         "Storage account (base)": 150}
USAGE = {"Model input tokens":  {"per": 1000, "rate": 0.22, "qty_per_claim": 3200},
         "Model output tokens": {"per": 1000, "rate": 0.88, "qty_per_claim": 600},
         "Function executions": {"per": 1000000, "rate": 17.0, "qty_per_claim": 9},
         "Cosmos RU overage":   {"per": 1000000, "rate": 21.0, "qty_per_claim": 240}}
CLAIMS_PER_MONTH = 12000
EXCLUSIONS = ["Data egress if the client moves to a multi-region design",
              "Storage growth - documents accumulate and are never deleted",
              "Log Analytics ingestion above the free tier, which grows with traffic",
              "Any reserved-instance or enterprise agreement discount",
              "Non-production environments (dev and staging are extra)",
              "Support plan"]

def variable_cost(claims):
    return {k: (v["qty_per_claim"] * claims / v["per"]) * v["rate"] for k, v in USAGE.items()}

fixed_total = sum(FIXED.values())
print("1. THE FLOOR - what it costs whether or not anybody uses it")
print("-" * 64)
for k, v in FIXED.items(): print("   " + k.ljust(42) + ("Rs " + format(round(v), ",")).rjust(14))
print("   " + "MONTHLY FLOOR".ljust(42) + ("Rs " + format(round(fixed_total), ",")).rjust(14))

for label, claims in [("2. AT YOUR STATED VOLUME", CLAIMS_PER_MONTH),
                      ("3. AT THREE TIMES THAT VOLUME", CLAIMS_PER_MONTH * 3)]:
    var = variable_cost(claims)
    print()
    print(label + " - " + format(claims, ",") + " claims/month")
    print("-" * 64)
    for k, v in var.items(): print("   " + k.ljust(42) + ("Rs " + format(round(v), ",")).rjust(14))
    tot = fixed_total + sum(var.values())
    print("   " + "TOTAL".ljust(42) + ("Rs " + format(round(tot), ",")).rjust(14))
    print("   " + ("variable share: " + str(round(100 * sum(var.values()) / tot)) + "%").rjust(56))

print()
print("EXPLICITLY EXCLUDED - say this out loud, do not bury it")
print("-" * 64)
for e in EXCLUSIONS: print("   -", e)
print()
print("Look at the variable share. For an AI system it is usually the larger number,")
print("and none of it is in your Bicep file. A client given one confident figure will")
print("budget from it. A client given the shape will forgive a wrong number.")

## Part 8 · Deliverables

### Cell 18 — Export the ADR and the cost sheet

Two files come out of today:

1. **`cost_model.csv`** — the three-figure model, which the client can argue with. That is the point; a model they argue with is a model they understand.
2. **`ADR-043-iac.md`** — the decisions, including the drift you did **not** close. Naming an accepted limitation is stronger than claiming there are none.

In [ ]:
import csv, datetime

with open("cost_model.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["section", "item", "monthly_inr", "note"])
    for k, v in FIXED.items(): w.writerow(["floor", k, round(v), "fixed, in the Bicep file"])
    for lbl, mult in [("stated_volume", 1), ("three_x_volume", 3)]:
        for k, v in variable_cost(CLAIMS_PER_MONTH * mult).items():
            w.writerow([lbl, k, round(v), "usage-driven, NOT in the Bicep file"])
    for e in EXCLUSIONS: w.writerow(["excluded", e, "", "not priced"])

adr = ["# ADR-043 - Infrastructure as code for DecisionStream", "",
 "Date: " + str(datetime.date.today()), "Author: <your name>", "Status: Proposed", "",
 "## Decision", "",
 "1. Three Bicep modules split by LIFECYCLE, not by technology:",
 "   observability (changes rarely, deployed first), data (changes rarely,",
 "   holds claims), compute (changes weekly). This keeps the blast radius of a",
 "   routine Function change away from the claims database.",
 "2. Environments differ by .bicepparam files only - one template, one code path.",
 "   Shape differences between dev and prod are listed below, not left implicit.",
 "3. Linter rules set to error for: secure-parameter-default, no-unused-params,",
 "   outputs-should-not-contain-secrets, no-hardcoded-env-urls.",
 "4. No secret is ever a deployment output. Deployment history is plain text and",
 "   readable by anyone with Reader for months; rotation does not clean it.",
 "   Secrets reach the app through Key Vault references, using managed identity.",
 "5. what-if runs on every pull request and is MACHINE-filtered. The pipeline fails",
 "   on Delete and on immutable-property changes. Noise is never shown to a human,",
 "   because a check people skim is not a check.", "",
 "## Accepted limitations - we are not pretending these are solved", "",
 "- Incremental mode never deletes, so the template guarantees a minimum, not the",
 "  whole. Mitigated by a nightly template-versus-cloud comparison, not eliminated.",
 "  Recommend deployment stacks with denySettings to close this properly.",
 "- what-if cannot evaluate listKeys(), reference(), secure params or utcNow().",
 "  Those paths are invisible to preview and are covered by tests instead.",
 "- The cost model's variable half is a MODEL, not a measurement. It should be",
 "  re-based against the first real month of invoices.", "",
 "## Cost summary", ""]
adr.append("- Monthly floor: Rs " + format(round(fixed_total), ","))
for lbl, mult in [("at stated volume", 1), ("at 3x volume", 3)]:
    t = fixed_total + sum(variable_cost(CLAIMS_PER_MONTH * mult).values())
    adr.append("- Total " + lbl + ": Rs " + format(round(t), ","))
adr += ["", "Excluded from all figures:"] + ["- " + e for e in EXCLUSIONS]
adr += ["", "## Open items", "", "| Item | Owner | Decide by |", "|---|---|---|",
        "| Deployment stacks vs incremental | <name> | <date> |",
        "| Function plan SKU (VNet needs Flex/Premium, not Y1) | <name> | <date> |"]

open("ADR-043-iac.md", "w").write(chr(10).join(adr))
print("Written:  cost_model.csv   ADR-043-iac.md")
print("In Colab, click the folder icon on the left to download them.\n")
print(chr(10).join(adr[:22]))
print("...")

## Notes — good to check last, after doing your exercise

Read this once you have run everything and written your ADR.

**1. Two cells were meant to fail.** Cell 6 (the linter) and Cell 8 (the secret output). If both went green on your first run, something did not execute — go back, because the failures are the lesson.

**2. The secret-output test is the one to take to every client.** It is nine lines of Python, it runs in a second, and it catches the most common credential leak in Azure deployments. Put it in the pipeline on day one of an engagement, before you write any Bicep of your own. You will find something.

**3. Your what-if filter is worth more than your Bicep.** Anybody can write Bicep, and the AI assistants write it well. Very few teams have made their what-if output *readable*, and the ones who have not are running a check that only appears to happen.

**4. Go back to the question from the deck.** *The resource group is deleted tonight — how long, and what would be missing?* The honest answer today is: the template covers the resource types you listed, you do not yet have deployment stacks, and Part 6 told you what is running that your template does not know about. That is a much better answer than "a few hours, we have the Bicep", and it is one you can actually defend.

**5. Things that will still be true in a year.** Tooling changes; these do not.

- A Bicep file is a statement of intent. The cloud is a separate thing that may or may not match it.
- Incremental mode never deletes, so "all provisioned by code" is a claim you keep proving.
- A test on a compiled template proves you generated what you meant — never that it will work.
- Never put a secret in an output. The history outlives the rotation.
- When a check is noisy, do not ask people to try harder. Narrow the check.
- The monthly cost of an AI system is mostly not in the infrastructure file.

---

*Confidential · For Learner Use Only · Birlasoft FORGE FDE Academy · Sprint 3 · 30 September 2026*